# NODES 2025 Demo

This notebook represents the demo part of the [presentation](https://neo4j.com/nodes-2025/agenda/easy-jupyter-notebook-graph-visualization-in-10-minutes/) at the NODES 2025 conference.

## Setup

In [ ]:
%pip install "neo4j-viz[gds, neo4j]" python-dotenv requests

In [ ]:
from dotenv import load_dotenv

# Load credentials for the neo4j database
load_dotenv("db_creds.env")

## Building our visualization graph



### From a Create Query string

The most simple way to test out neo4j-viz is by using `from_gql_create`.
The nodes and relationships are directly parsed from the provided query string. 

In [ ]:
from neo4j_viz.gql_create import from_gql_create

VG = from_gql_create("""
    CREATE
    (alice:Person {name: 'Alice', age: 30}),
    (bob:Person {name: 'Bob', age: 25}),
    (carol:Person {name: 'Carol', age: 27}),
    (alice)-[:FRIENDS_WITH {since: 2015}]->(bob),
    (bob)-[:FRIENDS_WITH {since: 2018}]->(carol)
""")

VG.render(initial_zoom=1.5)

## From a Neo4j database

Now lets assume, you have a Neo4j database available which you want to inspect.
In the following, I assume the Movies dataset is imported. You can use `:play movies` in the Neo4j Browser to import the dataset. 

In [ ]:
import os
import neo4j
from neo4j_viz.neo4j import from_neo4j

driver = neo4j.GraphDatabase.driver(
    uri=os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD")),
)

In [ ]:
from neo4j import Result

node_count = driver.execute_query(
    "MATCH (n) RETURN count(n) as count", result_transformer_=Result.to_df
).iloc[0, 0]
if node_count == 0:
    import requests

    raw_create_queries: str = requests.get(
        "https://raw.githubusercontent.com/neo4j-graph-examples/movies/refs/heads/main/scripts/movies.cypher"
    ).text
    movies_create_queries = [q for q in raw_create_queries.split(";") if q.strip()]
    for q in movies_create_queries:
        driver.execute_query(q)
    print("Database was empty. Demo graph created")

In [ ]:
# Limiting to 20 rows for demo purposes
VG = from_neo4j(driver, row_limit=20)
VG.render(initial_zoom=1.0)

## From GDS

In [ ]:
from graphdatascience import GraphDataScience

gds = GraphDataScience(
    endpoint=os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD")),
)
gds.set_database("neo4j")

In [ ]:
G, _ = gds.graph.cypher.project(
    query="""
    MATCH (s:Person)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(t:Person)
    WITH s, t, count(m) as common_movies
    WHERE s < t
    RETURN gds.graph.project('demo-graph', s, t, {
                                sourceNodeLabels: labels(s),
                                targetNodeLabels: labels(t),
                                relationshipProperties: {weight: common_movies},
                                relationshipType: 'CO_ACTED'
                }, {undirectedRelationshipTypes: ['CO_ACTED']})
"""
)

str(G)

How do we make sure the projection matches our expectation? Especially if the Cypher queries get more complex

In [ ]:
from neo4j_viz.gds import from_gds

# Limit to a couple of nodes to inspect the projection. Sampling via random-walks implemented in GDS
VG = from_gds(gds, G, db_node_properties=["name"], max_node_count=20)
VG.render()

Lets run some GDS algorithms to get some more insights about our co-actor graph.

In [ ]:
gds.pageRank.mutate(G, relationshipWeightProperty="weight", mutateProperty="pageRank")
gds.leiden.mutate(G, mutateProperty="component", relationshipWeightProperty="weight")

Other supported datasources include `from_snowflake`, and `from_pandas`.

## Customizing the Visualization

In [ ]:
from neo4j_viz.gds import from_gds

# make sure to include also the newly computed properties such as pageRank
# by default from_gds includes all properties of G
VG = from_gds(gds, G, db_node_properties=["name"])
VG.render()

In [ ]:
# Lets first fix the caption of the nodes to show the name.
for node in VG.nodes:
    node.caption = node.properties.get("name")

VG.render()

In [ ]:
# Inspect the computed communities to see which actors are grouped together
VG.color_nodes(property="component", override=True)
VG.render()

In [ ]:
# Resize nodes based on pageRank property, i.e., more important actors appear larger
VG.resize_nodes(property="pageRank")
VG.render()

In [ ]:
# To make sure certain nodes are always visible, we can pin them. Here we pin "Keanu Reeves".
pinned_nodes = {
    node.id: True
    for node in VG.nodes
    if node.properties.get("name") in ["Keanu Reeves"]
}

VG.toggle_nodes_pinned(pinned_nodes)
VG.render()

Further customization ideas to explore: 

* Modify the layout such as by adding coordinates
* Use custom colors from [`palettable.wesanderson`](https://jiffyclub.github.io/palettable/) 
* Change the size range for nodes

## Saving the Visualization

In [ ]:
# Use the save button in the rendered view. This produces a static image.
VG.render()

In [ ]:
# Save the raw HTML output to a file. This allows to share the interactive visualization.

import os
from neo4j_viz.options import Renderer

os.makedirs("./out", exist_ok=True)

# Save the visualization to a file
with open("out/co_acted.html", "w") as f:
    f.write(VG.render(renderer=Renderer.CANVAS).data)

## Cleanup

In [ ]:
driver.close()

In [ ]:
gds.graph.get("demo-graph").drop()

In [ ]:
gds.close()